# Proyecto Integrador de Minería de Datos I  
## Notebook 05 — Conclusiones integradas

**Carrera:** Tecnicatura Superior en Ciencia de Datos e Inteligencia Artificial  
**Asignatura:** Minería de Datos I  
**Dataset:** usuarios de una plataforma de streaming  
**Integrantes:** Thir Ferreyra Nadia Lorena y Constantinidi Leandro Exequiel  
**Comisión:** Turno Tarde  
**Fecha de cierre analítico:** 27 de junio de 2026  

---

## Propósito de esta etapa

Este notebook integra las evidencias obtenidas durante:

1. la inspección inicial;
2. la limpieza y validación;
3. el análisis exploratorio;
4. el escalamiento y PCA.

Las conclusiones se presentan diferenciando:

> **Evidencia → interpretación → conclusión**

También se explicitan las limitaciones y las mejoras futuras. Una limitación no es un hallazgo: es una condición que restringe el alcance de lo que puede afirmarse.

No se incluyen modelos predictivos ni se realizan afirmaciones causales.


## 1. Objetivo general del proyecto

Analizar las características y los patrones de comportamiento de los usuarios de una plataforma de streaming, evaluando cómo el plan de suscripción, la edad, el país y las preferencias de contenido se relacionan con el tiempo mensual de visualización, el uso del soporte y la actividad reciente.

## Preguntas de análisis

1. ¿Cómo se distribuyen los usuarios entre los planes de suscripción?
2. ¿Cómo se distribuye el tiempo mensual de visualización?
3. ¿El tiempo mensual cambia según el plan contratado?
4. ¿Existe una relación entre la edad y el tiempo mensual?
5. ¿El patrón de consumo por plan se mantiene al considerar el país?
6. ¿Las variables numéricas pueden resumirse eficientemente mediante PCA?


## 2. Importación de librerías

Este notebook vuelve a calcular los indicadores centrales a partir del dataset procesado. De esta manera, las conclusiones no dependen de copiar manualmente resultados de otros notebooks.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 3. Carga de datos y trazabilidad

Se cargan:

- el dataset original, únicamente para comprobar sus dimensiones;
- el dataset procesado, utilizado en el análisis;
- el log ETL, que documenta las transformaciones.

El archivo original no se modifica.


In [2]:
rutas_raw = [
    Path("../data/raw/streaming_users_dirty.json"),
    Path("data/raw/streaming_users_dirty.json"),
]

rutas_processed = [
    Path("../data/processed/streaming_users_clean.csv"),
    Path("data/processed/streaming_users_clean.csv"),
]

rutas_log = [
    Path("../logs/pipeline_log.csv"),
    Path("logs/pipeline_log.csv"),
]

ruta_raw = next((ruta for ruta in rutas_raw if ruta.exists()), None)
ruta_processed = next(
    (ruta for ruta in rutas_processed if ruta.exists()),
    None,
)
ruta_log = next((ruta for ruta in rutas_log if ruta.exists()), None)

if None in [ruta_raw, ruta_processed, ruta_log]:
    raise FileNotFoundError(
        "No se encontraron todos los archivos requeridos."
    )

df_raw = pd.read_json(ruta_raw)

df = pd.read_csv(
    ruta_processed,
    parse_dates=["last_login_date"],
)

pipeline_log = pd.read_csv(ruta_log)

df_control = df.copy(deep=True)

print(f"Dataset original: {df_raw.shape}")
print(f"Dataset procesado: {df.shape}")
print(f"Pasos registrados en el log: {len(pipeline_log)}")

Dataset original: (8160, 8)
Dataset procesado: (8000, 8)
Pasos registrados en el log: 11


## 4. Síntesis de calidad y preparación

La preparación no tuvo como objetivo eliminar todos los valores atípicos o nulos. Cada decisión buscó conservar información válida y corregir solamente situaciones respaldadas por evidencia.


In [3]:
resumen_calidad = pd.DataFrame({
    "indicador": [
        "Filas originales",
        "Filas procesadas",
        "Retención de filas",
        "Identificadores duplicados iniciales adicionales",
        "Identificadores duplicados finales",
        "Nulos iniciales totales",
        "Nulos finales totales",
        "Variable con nulos finales",
        "Planes finales",
        "Países finales",
        "Géneros finales incluyendo No informado",
    ],
    "resultado": [
        len(df_raw),
        len(df),
        f"{len(df) / len(df_raw) * 100:.2f}%",
        int(df_raw.duplicated(subset="user_id").sum()),
        int(df["user_id"].duplicated().sum()),
        int(df_raw.isna().sum().sum()),
        int(df.isna().sum().sum()),
        "last_login_date",
        int(df["subscription_plan"].nunique()),
        int(df["country"].nunique()),
        int(df["favorite_genre"].nunique()),
    ],
})

display(resumen_calidad)

print("\nEstado final del pipeline:")
display(pipeline_log.tail(1))

,indicador,resultado
0,Filas originales,8160
1,Filas procesadas,8000
2,Retención de filas,98.04%
3,Identificadores duplicados iniciales adicionales,160
4,Identificadores duplicados finales,0
5,Nulos iniciales totales,753
6,Nulos finales totales,501
7,Variable con nulos finales,last_login_date
8,Planes finales,3
9,Países finales,7



Estado final del pipeline:


,Paso,Descripción,Filas,Nulos,Retención (%)
10,10,"Convertir fechas válidas y conservar como faltantes las ambiguas, inválidas o futuras",8000,501,98.0400


### Evidencia

- El dataset pasó de 8.160 filas a 8.000 usuarios únicos.
- La retención fue del 98,04 %.
- Las únicas filas eliminadas fueron 160 apariciones posteriores de identificadores ya existentes.
- Las categorías se unificaron sin eliminar usuarios.
- Los valores numéricos imposibles fueron tratados mediante criterios documentados.
- Los 501 nulos finales pertenecen exclusivamente a fechas que no podían reconstruirse con seguridad.

### Interpretación

El proceso priorizó la conservación de usuarios y la trazabilidad. Tener 501 fechas faltantes al final no significa que el dataset esté mal limpiado: representa una decisión explícita de no inventar información.

### Conclusión de calidad

> El dataset procesado es adecuado para responder las preguntas descriptivas seleccionadas, siempre que se reconozcan las decisiones de imputación y la ausencia remanente en la fecha del último ingreso.


# 5. Respuestas a las preguntas del análisis exploratorio

Los siguientes resultados se recalculan directamente desde `streaming_users_clean.csv`.


In [4]:
orden_planes = ["Básico", "Estándar", "Premium"]

conteo_planes = (
    df["subscription_plan"]
    .value_counts()
    .reindex(orden_planes)
)

porcentaje_planes = conteo_planes / len(df) * 100

resumen_planes = pd.DataFrame({
    "usuarios": conteo_planes,
    "porcentaje": porcentaje_planes,
})

estadisticas_tiempo = pd.Series({
    "media": df["monthly_watch_time_mins"].mean(),
    "mediana": df["monthly_watch_time_mins"].median(),
    "desviacion_estandar": df["monthly_watch_time_mins"].std(),
    "coeficiente_variacion_pct": (
        df["monthly_watch_time_mins"].std()
        / df["monthly_watch_time_mins"].mean()
        * 100
    ),
    "asimetria": df["monthly_watch_time_mins"].skew(),
    "minimo": df["monthly_watch_time_mins"].min(),
    "maximo": df["monthly_watch_time_mins"].max(),
})

consumo_plan = (
    df.groupby("subscription_plan")["monthly_watch_time_mins"]
    .agg(["count", "mean", "median", "std"])
    .reindex(orden_planes)
)

correlacion_edad_tiempo = (
    df[["age", "monthly_watch_time_mins"]]
    .corr()
    .iloc[0, 1]
)

consumo_pais_plan = (
    df.pivot_table(
        index="country",
        columns="subscription_plan",
        values="monthly_watch_time_mins",
        aggfunc="mean",
    )
    .reindex(columns=orden_planes)
)

print("Distribución por plan:")
display(resumen_planes.round(2))

print("\nEstadísticas del tiempo mensual:")
display(estadisticas_tiempo.to_frame("valor").round(2))

print("\nConsumo por plan:")
display(consumo_plan.round(2))

print(
    "\nCorrelación entre edad y tiempo mensual: "
    f"{correlacion_edad_tiempo:.6f}"
)

print("\nConsumo medio por país y plan:")
display(consumo_pais_plan.round(1))

Distribución por plan:


,usuarios,porcentaje
subscription_plan,,
Básico,3600,45.0000
Estándar,2817,35.2100
Premium,1583,19.7900



Estadísticas del tiempo mensual:


,valor
media,800.8800
mediana,770.8500
desviacion_estandar,490.2700
coeficiente_variacion_pct,61.2200
asimetria,2.4300
minimo,0.0000
maximo,"4,193.7000"



Consumo por plan:


,count,mean,median,std
subscription_plan,,,,
Básico,3600,597.0000,552.7000,458.9700
Estándar,2817,871.7000,840.0000,436.5900
Premium,1583,"1,138.5300","1,127.0000",423.3400



Correlación entre edad y tiempo mensual: 0.006500

Consumo medio por país y plan:


subscription_plan,Básico,Estándar,Premium
country,,,
Argentina,606.3000,879.6000,"1,154.6000"
Brasil,615.7000,867.0000,"1,168.8000"
Chile,570.5000,853.5000,"1,174.5000"
Colombia,598.9000,872.3000,"1,088.7000"
México,593.3000,874.2000,"1,108.9000"
Perú,605.5000,892.1000,"1,138.0000"
Uruguay,590.2000,864.2000,"1,137.2000"


## Pregunta 1 — ¿Cómo se distribuyen los usuarios entre los planes?

### Evidencia

- Básico: 3.600 usuarios, 45 %.
- Estándar: 2.817 usuarios, 35,21 %.
- Premium: 1.583 usuarios, 19,79 %.

### Interpretación

La base se concentra en Básico y Estándar. Premium es el grupo menos numeroso, por lo que las comparaciones de comportamiento deben utilizar proporciones, medias o medianas y no solamente cantidades absolutas.

### Conclusión

> Aproximadamente ocho de cada diez usuarios pertenecen a los planes Básico o Estándar.


## Pregunta 2 — ¿Cómo se distribuye el tiempo mensual?

### Evidencia

- Media: aproximadamente 800,9 minutos.
- Mediana: aproximadamente 770,9 minutos.
- Desviación estándar: aproximadamente 490,3 minutos.
- Coeficiente de variación: superior al 60 %.
- Asimetría positiva: aproximadamente 2,43.

### Interpretación

El consumo es heterogéneo y presenta una cola de usuarios intensivos. La media está influida por esos valores altos y queda por encima de la mediana.

### Conclusión

> La mediana representa mejor el centro habitual del consumo, mientras que la media debe interpretarse junto con la dispersión.


## Pregunta 3 — ¿El consumo cambia según el plan?

### Evidencia

Las medianas son:

- Básico: 552,7 minutos.
- Estándar: 840 minutos.
- Premium: 1.127 minutos.

Los promedios siguen el mismo orden:

- Básico: aproximadamente 597 minutos.
- Estándar: aproximadamente 872 minutos.
- Premium: aproximadamente 1.139 minutos.

### Interpretación

Existe una diferencia clara y ordenada entre los grupos. Sin embargo, las distribuciones se superponen: el plan no determina completamente el comportamiento individual.

### Conclusión

> El plan de suscripción presenta la asociación descriptiva más clara con el nivel de consumo mensual, pero el análisis no demuestra que el plan cause ese consumo.


## Pregunta 4 — ¿La edad se relaciona con el consumo?

### Evidencia

La correlación de Pearson entre edad y minutos mensuales es aproximadamente 0,0065.

### Interpretación

El valor es prácticamente cero y no se observa una relación lineal relevante. Una correlación baja no descarta toda posible relación no lineal, pero sí indica que la edad aislada explica muy poco del comportamiento de consumo.

### Conclusión

> La edad no constituye una variable útil por sí sola para caracterizar el tiempo mensual de visualización en este dataset.


## Pregunta 5 — ¿El patrón por plan se mantiene entre países?

### Evidencia

En Argentina, Brasil, Chile, Colombia, México, Perú y Uruguay se repite el orden:

> Premium > Estándar > Básico

### Interpretación

El país introduce variaciones moderadas dentro de cada plan, pero no revierte el patrón general. La asociación entre plan y consumo no está producida únicamente por la composición de un país específico.

### Conclusión

> El patrón de mayor consumo en planes superiores es consistente en los siete países disponibles.


## Matriz integrada: evidencia, interpretación y conclusión


In [5]:
matriz_respuestas = pd.DataFrame({
    "pregunta": [
        "Distribución por plan",
        "Distribución del tiempo mensual",
        "Consumo según plan",
        "Edad y consumo",
        "País, plan y consumo",
    ],
    "evidencia": [
        "Básico 45%; Estándar 35,21%; Premium 19,79%",
        "Media 800,9; mediana 770,9; asimetría positiva",
        "Medianas: 552,7; 840; 1.127 minutos",
        "Correlación de Pearson ≈ 0,0065",
        "Premium > Estándar > Básico en los siete países",
    ],
    "interpretacion": [
        "La muestra se concentra en Básico y Estándar",
        "Existe heterogeneidad y una cola de usuarios intensivos",
        "Los planes superiores presentan mayor consumo grupal",
        "No existe una asociación lineal relevante",
        "El país modifica niveles, pero no el orden entre planes",
    ],
    "conclusion": [
        "Premium es el grupo menos numeroso",
        "La mediana describe mejor el centro habitual",
        "El plan es la variable más asociada con el consumo",
        "La edad aislada no caracteriza el consumo",
        "El patrón por plan es geográficamente consistente",
    ],
})

display(matriz_respuestas)

,pregunta,evidencia,interpretacion,conclusion
0,Distribución por plan,"Básico 45%; Estándar 35,21%; Premium 19,79%",La muestra se concentra en Básico y Estándar,Premium es el grupo menos numeroso
1,Distribución del tiempo mensual,"Media 800,9; mediana 770,9; asimetría positiva",Existe heterogeneidad y una cola de usuarios intensivos,La mediana describe mejor el centro habitual
2,Consumo según plan,"Medianas: 552,7; 840; 1.127 minutos",Los planes superiores presentan mayor consumo grupal,El plan es la variable más asociada con el consumo
3,Edad y consumo,"Correlación de Pearson ≈ 0,0065",No existe una asociación lineal relevante,La edad aislada no caracteriza el consumo
4,"País, plan y consumo",Premium > Estándar > Básico en los siete países,"El país modifica niveles, pero no el orden entre planes",El patrón por plan es geográficamente consistente


# 6. Conclusión de PCA

PCA se aplicó sobre:

- edad;
- minutos mensuales;
- tickets de soporte;
- días desde el último ingreso.

Se utilizaron 7.499 usuarios con fecha válida y se aplicó `StandardScaler`.


In [6]:
fecha_referencia = pd.Timestamp("2026-06-27")

df_pca = df.dropna(subset=["last_login_date"]).copy()

df_pca["days_since_last_login"] = (
    fecha_referencia - df_pca["last_login_date"]
).dt.days

variables_pca = [
    "age",
    "monthly_watch_time_mins",
    "customer_support_tickets",
    "days_since_last_login",
]

X = df_pca[variables_pca]

X_scaled = StandardScaler().fit_transform(X)

pca = PCA().fit(X_scaled)

tabla_pca = pd.DataFrame({
    "componente": [f"PC{i + 1}" for i in range(4)],
    "varianza_individual_pct": (
        pca.explained_variance_ratio_ * 100
    ),
    "varianza_acumulada_pct": (
        np.cumsum(pca.explained_variance_ratio_) * 100
    ),
})

display(tabla_pca.round(2))

print(
    f"Retención de casos para PCA: "
    f"{len(df_pca) / len(df) * 100:.2f}%"
)

,componente,varianza_individual_pct,varianza_acumulada_pct
0,PC1,25.4300,25.4300
1,PC2,25.0200,50.4500
2,PC3,24.8700,75.3200
3,PC4,24.6800,100.0000


Retención de casos para PCA: 93.74%


### Evidencia

- PC1: 25,43 %.
- PC1 + PC2: 50,45 %.
- PC1 + PC2 + PC3: 75,32 %.
- Se necesitan cuatro componentes para superar el 80 %.

### Interpretación

Las variables tienen correlaciones bajas y aportan dimensiones relativamente diferentes. No existe una dirección principal que concentre gran parte de la variabilidad.

### Conclusión de PCA

> PCA es útil para explorar la estructura, pero no permite reducir las cuatro variables a dos o tres componentes sin una pérdida importante de información.

Conservar las variables originales resulta más claro para la comunicación y mantiene toda la información disponible.


# 7. Cumplimiento de los objetivos

La siguiente tabla relaciona cada objetivo con la evidencia que demuestra su cumplimiento.


In [7]:
cumplimiento_objetivos = pd.DataFrame({
    "objetivo": [
        "Comprender la estructura y calidad inicial",
        "Preparar datos con decisiones justificadas",
        "Realizar análisis univariado, bivariado y multivariado",
        "Aplicar escalamiento y PCA",
        "Mantener trazabilidad y reproducibilidad",
        "Comunicar conclusiones coherentes",
    ],
    "evidencia_de_cumplimiento": [
        "Notebook 01: dimensiones, tipos, nulos, duplicados y valores sospechosos",
        "Notebook 02: evidencia, acción, impacto, dataset procesado y log ETL",
        "Notebook 03: cinco visualizaciones vinculadas con preguntas",
        "Notebook 04: selección de variables, StandardScaler, varianza y loadings",
        "Dataset raw preservado, dataset processed, notebooks ejecutables y pipeline_log.csv",
        "Notebook 05: respuestas, limitaciones, mejoras y alcance",
    ],
    "estado": [
        "Cumplido",
        "Cumplido",
        "Cumplido",
        "Cumplido",
        "Cumplido",
        "Cumplido",
    ],
})

display(cumplimiento_objetivos)

,objetivo,evidencia_de_cumplimiento,estado
0,Comprender la estructura y calidad inicial,"Notebook 01: dimensiones, tipos, nulos, duplicados y valores sospechosos",Cumplido
1,Preparar datos con decisiones justificadas,"Notebook 02: evidencia, acción, impacto, dataset procesado y log ETL",Cumplido
2,"Realizar análisis univariado, bivariado y multivariado",Notebook 03: cinco visualizaciones vinculadas con preguntas,Cumplido
3,Aplicar escalamiento y PCA,"Notebook 04: selección de variables, StandardScaler, varianza y loadings",Cumplido
4,Mantener trazabilidad y reproducibilidad,"Dataset raw preservado, dataset processed, notebooks ejecutables y pipeline_log.csv",Cumplido
5,Comunicar conclusiones coherentes,"Notebook 05: respuestas, limitaciones, mejoras y alcance",Cumplido


# 8. Implicaciones descriptivas

Las siguientes implicaciones se limitan a lo que los datos permiten sostener:

1. **Segmentación por plan:** el plan es más informativo que la edad para describir diferencias de consumo.
2. **Uso de mediana y dispersión:** los reportes deberían mostrar la mediana junto con la media debido a la asimetría del consumo.
3. **País como dimensión secundaria:** conviene permitir filtros por país, pero el patrón principal continúa organizado por plan.
4. **Usuarios intensivos:** los consumos altos plausibles no deberían tratarse automáticamente como errores.
5. **PCA como exploración:** no conviene reemplazar las cuatro variables numéricas por solamente dos componentes.

Estas implicaciones no constituyen recomendaciones comerciales definitivas. Para decisiones de precios, promociones o retención se necesitarían variables adicionales y un diseño analítico específico.


# 9. Qué puede y qué no puede afirmarse

| Puede afirmarse | No puede afirmarse |
|---|---|
| Los planes presentan diferentes niveles medios y medianos de consumo | Contratar Premium causa mayor consumo |
| La edad tiene correlación lineal casi nula con los minutos | La edad nunca influye en el comportamiento |
| El orden por plan se repite en los siete países | El patrón representa a todos los países o plataformas |
| Los cuatro componentes son necesarios para superar el 80 % | PCA demuestra que las variables carecen de utilidad |
| El dataset procesado es coherente para este análisis descriptivo | El dataset está libre de toda incertidumbre o sesgo |

Esta distinción es esencial para mantener conclusiones proporcionales a la evidencia.


# 10. Limitaciones

Las limitaciones restringen el alcance del proyecto; no son resultados favorables ni desfavorables.

## Limitaciones del dataset

1. No se dispone de documentación externa sobre el método de muestreo, la población total ni el período exacto al que corresponde el consumo mensual.
2. Faltan variables potencialmente relevantes: precio, antigüedad de la cuenta, cantidad de perfiles, dispositivos, renovaciones, cancelaciones y contenido efectivamente reproducido.
3. Se imputaron edades, minutos y tickets mediante criterios documentados. Toda imputación introduce cierto grado de incertidumbre.
4. Permanecen 501 fechas desconocidas porque no podían interpretarse sin realizar suposiciones.
5. El dataset contiene países latinoamericanos seleccionados y no representa necesariamente a una plataforma global.

## Limitaciones del proceso

1. El análisis es descriptivo y exploratorio; no se realizó inferencia causal.
2. La correlación de Pearson evalúa relaciones lineales y puede no detectar patrones no lineales.
3. PCA también resume principalmente estructura lineal.
4. PCA utilizó casos completos respecto de la fecha y retuvo el 93,74 % de los usuarios.
5. Los criterios de validez se definieron a partir de evidencia interna y límites lógicos, ya que no se proporcionó un diccionario oficial de reglas de negocio.

## Limitación de alcance

> Las conclusiones describen patrones del dataset analizado. No deben generalizarse automáticamente a otras plataformas, períodos o poblaciones.


# 11. Mejoras futuras

1. **Incorporar documentación de origen:** período de observación, procedimiento de muestreo y reglas de negocio.
2. **Agregar variables de comportamiento:** sesiones, dispositivos, perfiles, categorías vistas, finalización de contenidos y frecuencia semanal.
3. **Agregar variables comerciales:** precio pagado, descuentos, renovaciones, cancelaciones y antigüedad de la suscripción.
4. **Mejorar el control en origen:** validar rangos, categorías y formatos de fecha al momento de registrar los datos.
5. **Registrar versiones temporales:** incluir fecha de actualización para resolver duplicados que contengan información diferente.
6. **Analizar faltantes en profundidad:** investigar por qué algunas fechas y minutos no fueron registrados.
7. **Realizar análisis longitudinal:** observar cambios de consumo y plan a través del tiempo.
8. **Evaluar relaciones no lineales:** utilizar técnicas adicionales si el objetivo futuro lo justifica.
9. **Realizar análisis de sensibilidad:** comparar los resultados con otras estrategias de imputación y escalamiento.
10. **Desarrollar modelado predictivo en otro proyecto:** por ejemplo, predicción de abandono, solamente si se incorpora una variable objetivo adecuada.

Estas acciones amplían el proyecto, pero no son requisitos pendientes de esta etapa.


# 12. Conclusión general

El proyecto permitió transformar un dataset con duplicados, categorías inconsistentes, valores imposibles, faltantes y fechas problemáticas en una base de 8.000 usuarios únicos, conservando el 98,04 % de las filas originales y documentando cada transformación.

El análisis exploratorio mostró que el plan de suscripción es la variable que presenta la asociación más clara con el tiempo mensual de visualización. Las medianas aumentan de Básico a Estándar y Premium, y este orden se mantiene en los siete países. En cambio, la edad presenta una correlación lineal prácticamente nula con el consumo.

PCA confirmó que edad, consumo, soporte y recencia aportan información relativamente diferente. Los dos primeros componentes resumen solo el 50,45 % de la variabilidad y se requieren cuatro para superar el 80 %, por lo que no se recomienda reemplazar las variables originales mediante una reducción a dos o tres componentes.

La conclusión principal es:

> **El comportamiento de consumo se diferencia principalmente por plan, pero no puede explicarse completamente mediante las variables disponibles. La calidad de las conclusiones depende tanto de los patrones observados como de las decisiones documentadas y de reconocer las limitaciones del dataset.**


# 15. Validación final

Se comprueba la coherencia de los indicadores centrales y que el dataset procesado no haya sido modificado durante este notebook.


In [8]:
controles = {
    "Dataset procesado sin modificaciones": df.equals(df_control),
    "8.000 usuarios finales": len(df) == 8000,
    "0 identificadores duplicados": df["user_id"].duplicated().sum() == 0,
    "501 fechas faltantes": df["last_login_date"].isna().sum() == 501,
    "Básico representa 45%": np.isclose(
        porcentaje_planes.loc["Básico"],
        45.0,
    ),
    "Correlación edad-tiempo casi nula": abs(
        correlacion_edad_tiempo
    ) < 0.01,
    "PC1 y PC2 explican aproximadamente 50,45%": np.isclose(
        tabla_pca["varianza_acumulada_pct"].iloc[1],
        50.45124,
        atol=0.01,
    ),
    "Cuatro componentes explican 100%": np.isclose(
        tabla_pca["varianza_acumulada_pct"].iloc[-1],
        100.0,
    ),
}

tabla_controles = pd.DataFrame({
    "control": list(controles.keys()),
    "resultado": list(controles.values()),
})

display(tabla_controles)

if not all(controles.values()):
    fallidos = [
        nombre
        for nombre, resultado in controles.items()
        if not resultado
    ]
    raise AssertionError(
        f"Fallaron los controles finales: {fallidos}"
    )

print("Todas las conclusiones cuantitativas fueron verificadas.")

,control,resultado
0,Dataset procesado sin modificaciones,True
1,8.000 usuarios finales,True
2,0 identificadores duplicados,True
3,501 fechas faltantes,True
4,Básico representa 45%,True
5,Correlación edad-tiempo casi nula,True
6,"PC1 y PC2 explican aproximadamente 50,45%",True
7,Cuatro componentes explican 100%,True


Todas las conclusiones cuantitativas fueron verificadas.


## Cierre

Con este notebook queda completada la secuencia técnica obligatoria:

- `01_inspeccion_inicial.ipynb`
- `02_calidad_y_limpieza.ipynb`
- `03_eda.ipynb`
- `04_pca.ipynb`
- `05_conclusiones.ipynb`

Las próximas etapas corresponden a la comunicación del proyecto:

1. aplicación Streamlit;
2. README definitivo;
3. informe final de hasta dos páginas;
4. publicación en GitHub y Streamlit Cloud.
